# 01 — Data download and EDA

Pulls the two public datasets and does a quick exploratory look.

- **Synthetic / real CFRP**: Chalmers Zenodo record 10.5281/zenodo.15389426 (Friemann et al. 2025).
  The record contains the *real* reconstructed 3D-textile CFRP scan. The synthetic training pipeline is open-source but generated offline; we use our `synthetic.py` generator as a substitute.
- **SiC-SiC CMC**: Badran et al. on the Argonne ACDC portal. Distributed via Materials Data Facility; requires Globus, so the helper only drops a README.

If you are disk-constrained, skip the real download and use the synthetic generator only.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path('..').resolve()
RAW = REPO / 'data' / 'raw'
PROC = REPO / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

## 1. Chalmers synthetic CFRP (real scan)
~512 MB. Skip this cell if you already have the file in `data/raw/cfrp/`.

In [ ]:
from cfrp_medsam2.download import download_cfrp, list_cfrp_files

files = list_cfrp_files()
for f in files:
    print(f"{f['key']:60s} {f['size'] / 1024 / 1024:7.1f} MB")

paths = download_cfrp(RAW / 'cfrp')
print('downloaded:', paths)

## 2. SiC-SiC CMC hint file
The ACDC/MDF repo requires Globus auth; we can't pull it non-interactively, but we drop a README so the human operator can finish the download.

In [ ]:
from cfrp_medsam2.download import download_sic_sic_hint
print('hint at:', download_sic_sic_hint(RAW / 'sic_sic'))

## 3. EDA on the real CFRP scan
Inspect intensity distribution — the key difficulty is that fibre and matrix are almost indistinguishable by grey level alone.

In [ ]:
import tifffile

tiff = RAW / 'cfrp' / 'real_orthogonal_noobed_sample_reconstruction.tiff'
vol = tifffile.imread(tiff)
print('shape', vol.shape, 'dtype', vol.dtype, 'range', vol.min(), vol.max())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, z in zip(axes, [vol.shape[0] // 4, vol.shape[0] // 2, 3 * vol.shape[0] // 4]):
    ax.imshow(vol[z], cmap='gray', vmin=np.percentile(vol, 1), vmax=np.percentile(vol, 99))
    ax.set_title(f'z={z}')
    ax.axis('off')
plt.tight_layout(); plt.show()

flat = vol.ravel()
mask_sample = flat[(flat > np.percentile(flat, 1)) & (flat < np.percentile(flat, 99))]
plt.figure(figsize=(6, 3))
plt.hist(mask_sample, bins=200)
plt.title('intensity histogram (1–99 percentile)')
plt.xlabel('attenuation')
plt.ylabel('count')
plt.show()

## 4. Synthetic toy volume
Our toy generator mimics the key difficulty: fibre / matrix intensities differ by only ~8% of dynamic range.

In [ ]:
from cfrp_medsam2.synthetic import ToyVolumeConfig, make_toy_volume
imgs, gts = make_toy_volume(ToyVolumeConfig(shape=(16, 256, 256), num_fibres=40))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(imgs[8], cmap='gray'); axes[0].set_title('synthetic slice'); axes[0].axis('off')
axes[1].imshow(imgs[8], cmap='gray'); axes[1].imshow(gts[8], alpha=0.4, cmap='Reds')
axes[1].set_title('fibre mask overlay'); axes[1].axis('off')
plt.tight_layout(); plt.show()